In [1]:
import torch
print('PyTorch version:', torch.__version__)
from torch import nn
from torch import optim
from torch.utils.data import DataLoader

import torchvision
# print('Torchvision version:', torchvision.__version__)


import numpy as np

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device is:', device, '\n')

import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision import datasets
from torchvision.transforms import ToTensor

import matplotlib.pyplot as plt
import os
from PIL import Image



# from tqdm import tqdm
# from torchinfo import summary
# from torchvision.utils import make_grid
# import matplotlib.pyplot as plt
# from torchvision.io import decode_image


PyTorch version: 2.5.1+cu118
Device is: cpu 



Config Section

In [2]:
initial_channel_count = 1 #number of channels the initial image has, possibly add an output channel count if we want a different output
img_size = (128, 128)
images_path = 'mc_outputs'

UNet_params = {
    "scale_rate" : 2, #integer controlling the amount the spatial resolution of the feature map decreases/increases during the down/upsampling process
    "time_embedding_dimensions" : 128, #integer controlling dimensionality of positional embedding vector of the timesteps
    "num_conv_blocks_per_scaling_block" : 1, #integer that controls how many convolutional blocks there are per up/down sampling block
    "use_attention" : True,
}

diffusion_parameters = {
    'scheduling_type' : 'linear',
    "num_timesteps" : 1000,
    "max_variance" : 0.02,
}

training_params = {
    'batch_size' : 1,
    'num_epochs' : 20,
    'fmap_lossfunc': nn.MSELoss(),
    'sbit_lossfunc': nn.BCEWithLogitsLoss()
}


In [3]:
from diffusion_module import diffusion
from Unet_import import UNet

diffusion_module = diffusion(startVariance=0, maxVariance=diffusion_parameters['max_variance'], diffusionSteps=1000, device='cpu')
model = UNet(t_dim=UNet_params['time_embedding_dimensions'])
optimizer = optim.AdamW(model.parameters(), lr = 0.0001)


Training Cycle

In [4]:
from dataloading import MCImageDataset
from sparsity import create_sparsity_mask, get_sparsity_mask, induce_sparsity

dataset_path = images_path

training_dataset = MCImageDataset(dataset_path)
testing_dataset = MCImageDataset(dataset_path)

training_dataloader = DataLoader(training_dataset, batch_size=training_params['batch_size'], shuffle=True, drop_last=True)
test_dataloader = DataLoader(testing_dataset, batch_size=training_params['batch_size'], shuffle=False, drop_last=True)

num_epochs = 1
overall_losses = []
for epoch in range(num_epochs):
    losses = []
    for data in training_dataloader:
        model.train()
        data = data.to(device)

        sparsity_mask = create_sparsity_mask(data)
        data_and_mask = torch.cat([data, sparsity_mask], dim=1)
        
        t = torch.randint(low=0, high=diffusion_parameters['num_timesteps'], size=(data_and_mask.shape[0],)).to(device)
        noise = torch.randn_like(data_and_mask)
        
        x_t = diffusion_module.forward(input=data_and_mask, noise=noise, timestep=t)

        prediction = model(x_t, t)
        predicted_sparsity_mask, predicted_feature_map = get_sparsity_mask(prediction)
        print(f'mask size: {predicted_sparsity_mask.shape}')

        feature_map_loss = training_params['fmap_lossfunc'](data, predicted_feature_map)
        sbit_loss = training_params['sbit_lossfunc'](sparsity_mask, predicted_sparsity_mask)
        batch_loss = sbit_loss+feature_map_loss

        optimizer.zero_grad()
        batch_loss.backward()
        optimizer.step()


        losses.append(batch_loss.item())
        
    training_per_epoch_loss = np.array(losses).mean()
    overall_losses.append(training_per_epoch_loss)

    # print(training_per_epoch_loss)

print(overall_losses)



KeyError: 128